In [2]:
from time import time
import pod5
import numpy as np
import os
import pysam
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator
import matplotlib.colors as colors
import pandas as pd

In [3]:
working_dir = "./exemplar_figures/"

## Figure 4A

In [ ]:
def read_id_set(inpath):
    read_id_set = set()
    with open(inpath, 'r') as infile:
        for line in infile:
            read_id_set.add(line.strip())
    return read_id_set

def get_move_to_signal(mv_table_in, ts, ns):
    return np.append(np.where(np.array(mv_table_in, dtype="int8")==1)[0] * 6 + ts, ns)

def get_read_to_signal(mv_table_in, ts, ns, read_len):
    move_to_signal = get_move_to_signal(mv_table_in, ts, ns)
    
    readpos_to_signal = {}
    for i in range(read_len-2, -1, -1):
        readpos_to_signal[i] = (move_to_signal[read_len - i - 1], 
                                move_to_signal[read_len - i])
    return readpos_to_signal

def get_positions(read, low_pos, high_pos):
    read_pos_to_signal = get_read_to_signal(read.get_tag('mv'), read.get_tag('ts'), read.get_tag('ns'), len(read.query_sequence))
    ref_to_signal = {}
    for pair in read.get_aligned_pairs():
        if pair[1] is not None and low_pos < pair[1] < high_pos:
            if low_pos < pair[1] < high_pos and pair[0] is None:
                return None
            else:
                ref_to_signal[pair[1]] = read_pos_to_signal[pair[0]]
    return ref_to_signal

def pos_to_plot_index(max_index, read_data):
    read_data['x_axis'] = {}
    for i in read_data['pos_to_signal']:
        index = int(abs(max_index - i))
        total_units = len(read_data['pos_to_signal'][i])
        x_axis = [index + x/total_units for x in range(total_units)]
        read_data['x_axis'][i] = x_axis
    return read_data

def make_read_dict(bam_path, read_ids, pod5, target_position, lower_bound, upper_bound, max_reads=None):
    
    read_count = 0
    read_to_positions = {}
    
    for read in pysam.AlignmentFile(bam_path).fetch():
        if read.query_name not in read_ids or read.has_tag('pi'):
            continue
        
        x = get_positions(read, 
                          target_position - lower_bound, 
                          target_position + upper_bound)
        if x is not None:
            read_to_positions[read.query_name] = {'pos_to_signal':x}
            read_count += 1
        
        if max_reads is not None and read_count >= max_reads:
            return read_to_positions
    
    return read_to_positions

def update_to_signal(read_to_positions, pod5_file, max_pos):
    with pod5.DatasetReader(pod5_file, recursive = True) as dataset:
        for read_record in dataset:
            read_id = str(read_record.read_id)
            if read_id not in read_to_positions:
                continue
            for position in read_to_positions[read_id]['pos_to_signal']:
                start, stop = read_to_positions[read_id]['pos_to_signal'][position]
                read_to_positions[read_id]['pos_to_signal'][position] = read_record.signal_pa[start:stop]
    
    for read in list(read_to_positions.keys()):
        
        pos_to_plot_index(max_pos, read_to_positions[read])
    
    return read_to_positions

def plot_conditions(read_to_positions_list, labels_list, max_plot, target_length):
    palette = sns.color_palette('tab10')
    fig, ax = plt.subplots(1)
    fig.set_figwidth(4*target_length)
    fig.set_figheight(10)
    for j, read_to_positions in enumerate(read_to_positions_list):
        print(read_to_positions)
        count = 0
        #print(len(read_to_positions))
        for read_key in read_to_positions:
            if count > max_plot:
                break
            read = read_to_positions[read_key]
            for i in read['pos_to_signal']:
                sns.lineplot(x = read['x_axis'][i], 
                             y = read['pos_to_signal'][i], 
                             color = palette[j], 
                             alpha = 0.05,
                             linewidth=1)
            count += 1
                
    ax.set_ylim([40, 140])
    
    return fig, ax

In [ ]:
psi_1_mod_read_ids = read_id_set("./read_id_set_mod.txt")
psi_1_canon_read_ids = read_id_set("./read_id_set_canonical.txt")

psi_1_mod_bam = "/work/Genometechlab/stuart/whole_genome_ivt/04_10_25_RNA004_GM12878_wholeIVT.dorado_0.8.1.emit_moves.4mods.threshold_0.polya.GRCh38.sorted.filtered.bam"
psi_1_canon_bam = "/work/Genometechlab/stuart/whole_genome_ivt/04_10_25_RNA004_GM12878_wholeIVT.dorado_0.8.1.emit_moves.4mods.threshold_0.polya.GRCh38.sorted.filtered.bam"

psi_1_pod5 = "/work/Genometechlab/nanopore/PromethION/04_10_25_RNA004_GM12878_wholeIVT/04_10_25_RNA004_GM12878_wholeIVT/20250410_1517_2G_PBE01761_ca67591e/pod5_skip"

psi_1_mod_target_pos = 1370
psi_1_canon_target_pos = 1370

psi_1_lower_bound = 5
psi_1_upper_bound = 5
print("here1")
psi_1_read_canon_dict = make_read_dict(psi_1_canon_bam, psi_1_canon_read_ids, psi_1_pod5,  psi_1_mod_target_pos, psi_1_lower_bound, psi_1_upper_bound)
psi_1_read_canon_dict = update_to_signal(psi_1_read_canon_dict, psi_1_pod5, psi_1_mod_target_pos + psi_1_upper_bound)
print("here2")
psi_1_read_mod_dict = make_read_dict(psi_1_mod_bam, psi_1_mod_read_ids, psi_1_pod5,  psi_1_mod_target_pos, psi_1_lower_bound, psi_1_upper_bound)
psi_1_read_mod_dict = update_to_signal(psi_1_read_mod_dict, psi_1_pod5, psi_1_mod_target_pos + psi_1_upper_bound)

In [ ]:
pos_to_index = {1366:0,
                1367:1,
                1368:2,
                1369:3,
                1370:4,
                1371:5,
                1372:6,
                1373:7,
                1374:8
               }

In [ ]:
fig, ax = plot_conditions([psi_1_read_mod_dict, psi_1_read_canon_dict], ["Psi False Positive", "Canon"], 250, psi_1_upper_bound + psi_1_lower_bound + 1)
fig.get_figure().savefig(os.path.join(working_dir, "Psi_chrM_1370_signal.pdf"))

## Figure 4B

In [ ]:
df = pd.DataFrame.from_dict(psi_1_read_mod_dict, orient="index")
for key in psi_1_read_mod_dict:
    for pos in psi_1_read_mod_dict[key]['pos_to_signal']:
        for measurement in psi_1_read_mod_dict[key]['pos_to_signal'][pos]:
            if measurement > 250:
                print(psi_1_read_mod_dict[key]['pos_to_signal'])

In [ ]:
mod_medians = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}
mod_variance = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}
mod_dwell_time = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}

for key in psi_1_read_mod_dict:
    for pos in psi_1_read_mod_dict[key]['pos_to_signal']:
        mod_medians[pos_to_index[pos]].append(np.median(psi_1_read_mod_dict[key]['pos_to_signal'][pos]))
        mod_variance[pos_to_index[pos]].append(np.std(psi_1_read_mod_dict[key]['pos_to_signal'][pos]))
        mod_dwell_time[pos_to_index[pos]].append(len(psi_1_read_mod_dict[key]['pos_to_signal'][pos]))

In [ ]:
canon_medians = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}
canon_variance = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}
canon_dwell_time = {0:[], 1:[], 2:[], 3:[], 4:[], 5:[], 6:[], 7:[], 8:[]}

for key in psi_1_read_canon_dict:
    for pos in psi_1_read_canon_dict[key]['pos_to_signal']:
        canon_medians[pos_to_index[pos]].append(np.median(psi_1_read_canon_dict[key]['pos_to_signal'][pos]))
        canon_variance[pos_to_index[pos]].append(np.std(psi_1_read_canon_dict[key]['pos_to_signal'][pos]))
        canon_dwell_time[pos_to_index[pos]].append(len(psi_1_read_canon_dict[key]['pos_to_signal'][pos]))

In [ ]:
bwa = 2
fig, ax = plt.subplots(3, 9)
palette = sns.color_palette('tab10')
fig.set_figwidth(4*9)
fig.set_figheight(10)
for i in range(9):
    for j in range(3):
        if j == 0:
            sns.kdeplot(mod_medians[i], alpha = 0.25, color = palette[0], ax = ax[j][i], fill=True, bw_adjust = bwa)
            sns.kdeplot(canon_medians[i], alpha = 0.25, color = palette[1], ax = ax[j][i], fill=True, bw_adjust = bwa)
            ax[j][i].set_xlim([30,80])
            ax[j][i].set_ylim([0,0.18])
        if j == 1:
            sns.kdeplot(mod_variance[i], alpha = 0.25, color = palette[0], ax = ax[j][i], fill=True, bw_adjust = bwa)
            sns.kdeplot(canon_variance[i], alpha = 0.25, color = palette[1], ax = ax[j][i], fill=True, bw_adjust = bwa)
            ax[j][i].set_xlim([0, 10])
            ax[j][i].set_ylim([0, 1.1])
        if j == 2:
            sns.kdeplot(mod_dwell_time[i], alpha = 0.25, color = palette[0], ax = ax[j][i], fill=True, bw_adjust = bwa, clip=[0, 200])
            sns.kdeplot(canon_dwell_time[i], alpha = 0.25, color = palette[1], ax = ax[j][i], fill=True, bw_adjust = bwa, clip=[0, 200])
            ax[j][i].set_xlim([-5, 100])
            ax[j][i].set_ylim([0, 0.04])

fig.get_figure().savefig(os.path.join(working_dir, "signal_support.pdf"))